# 03 – Silver Transformation (Task C01 · C02 · C03)

**CommerceFlow 360** · Người phụ trách: **Duy Lê** · Mục báo cáo: 3.2, 3.4, 3.7

| Task | Nội dung | Đầu ra |
|---|---|---|
| C01 | Cài dbt Core + dbt-duckdb, khai báo `sources.yml` trỏ tới `bronze_*`, chạy `dbt debug` | `dbt_project/dbt_project.yml`, `dbt_project/models/sources.yml`, `dbt_project/profiles.yml.example` |
| C02 | 9 model `silver_*` cho 9 bảng Olist (+ `silver_holidays` hỗ trợ `dim_date`) | `dbt_project/models/silver/silver_*.sql` |
| C03 | dbt tests: unique, not_null, accepted_values, relationships + test tùy chỉnh | `dbt_project/models/silver/schema.yml`, `dbt_project/tests/test_silver_*.sql` |

**Đầu vào:** file `commerceflow.duckdb` đã có các bảng `bronze_*` do notebook `02_Bronze_Ingestion.ipynb` (Task B01) tạo ra.
Notebook chạy được trên Google Colab và máy cá nhân, không phụ thuộc đường dẫn riêng của thành viên nào.

## 1. Chuẩn bị môi trường

In [ ]:
# Trên Colab: clone repository (bỏ qua nếu đã đứng trong repository)
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not Path('/content/CommerceFlow360').exists():
    !git clone -q https://github.com/huyenpiano1711/CommerceFlow360.git /content/CommerceFlow360
if IN_COLAB:
    %cd /content/CommerceFlow360
print('Google Colab:', IN_COLAB)

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'dbt_project' / 'dbt_project.yml').exists() and (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc CommerceFlow360 (cần có dbt_project/dbt_project.yml).')

PROJECT_ROOT = find_project_root()
DBT_DIR = PROJECT_ROOT / 'dbt_project'
DUCKDB_PATH = Path(os.environ.get('DUCKDB_PATH', PROJECT_ROOT / 'commerceflow.duckdb')).resolve()
TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
DQ_DIR = PROJECT_ROOT / 'docs' / 'data_quality'
for d in (TABLE_DIR, DQ_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ['DUCKDB_PATH'] = str(DUCKDB_PATH)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DUCKDB_PATH :', DUCKDB_PATH)

Cài dbt Core và adapter dbt-duckdb (cú pháp `data_tests` / `arguments` trong `schema.yml` cần dbt ≥ 1.10).

In [ ]:
%pip install -q "dbt-core>=1.10,<2.0" "dbt-duckdb>=1.10,<2.0" "duckdb>=1.1,<2.0" pandas

In [ ]:
import subprocess, shutil, json
import duckdb, pandas as pd
pd.set_option('display.max_colwidth', 120)

def run_dbt(*args, check=True):
    """Chạy một lệnh dbt trong thư mục dbt_project và in toàn bộ log."""
    cmd = ['dbt', *args, '--no-use-colors']
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=DBT_DIR, capture_output=True, text=True, env=os.environ.copy())
    print(proc.stdout[-15000:])
    if proc.returncode != 0:
        print(proc.stderr[-5000:])
        if check:
            raise RuntimeError(f'dbt {args[0]} thất bại (mã {proc.returncode})')
    return proc

def query(sql):
    """Truy vấn chỉ đọc; đóng kết nối ngay để không khóa file khi dbt chạy."""
    with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
        return con.sql(sql).df()

print(subprocess.run(['dbt', '--version'], capture_output=True, text=True).stdout)

## 2. Kiểm tra tầng Bronze đầu vào

Silver đọc các bảng Bronze qua `source('bronze', ...)` khai báo trong `models/sources.yml`.
Cell dưới kiểm tra đủ 10 bảng `bronze_*`. Nếu thiếu, hãy chạy `02_Bronze_Ingestion.ipynb` trước
(hoặc tải bản sao lưu `commerceflow_YYYYMMDD.duckdb` rồi đổi tên thành `commerceflow.duckdb`).

In [ ]:
BRONZE_TABLES = ['bronze_customers', 'bronze_geolocation', 'bronze_order_items', 'bronze_order_payments',
                 'bronze_order_reviews', 'bronze_orders', 'bronze_products', 'bronze_sellers',
                 'bronze_product_category_name_translation', 'bronze_holidays']

def existing_tables():
    if not DUCKDB_PATH.exists():
        return set()
    return set(query("select table_name from information_schema.tables where table_schema = 'main'")['table_name'])

missing_bronze = [t for t in BRONZE_TABLES if t not in existing_tables()]
print('Bảng Bronze còn thiếu:', missing_bronze or 'không có')

**Tùy chọn – chỉ dùng khi chưa có kết quả Task B01.** Đặt `TAO_BRONZE_TAM = True` để nạp nhanh 10 CSV vào
`bronze_*` (giữ nguyên chuỗi, thêm `_ingested_at`, `_source_file`) nhằm kiểm thử Silver độc lập.
Bản Bronze chính thức vẫn là kết quả của `02_Bronze_Ingestion.ipynb`.

In [ ]:
TAO_BRONZE_TAM = False

RAW_FILES = {
    'bronze_customers': 'olist_customers_dataset.csv',
    'bronze_geolocation': 'olist_geolocation_dataset.csv',
    'bronze_order_items': 'olist_order_items_dataset.csv',
    'bronze_order_payments': 'olist_order_payments_dataset.csv',
    'bronze_order_reviews': 'olist_order_reviews_dataset.csv',
    'bronze_orders': 'olist_orders_dataset.csv',
    'bronze_products': 'olist_products_dataset.csv',
    'bronze_sellers': 'olist_sellers_dataset.csv',
    'bronze_product_category_name_translation': 'product_category_name_translation.csv',
    'bronze_holidays': 'holidays_brazil.csv',
}
if TAO_BRONZE_TAM and missing_bronze:
    with duckdb.connect(str(DUCKDB_PATH)) as con:
        for table in missing_bronze:
            src = (PROJECT_ROOT / 'data' / 'raw' / RAW_FILES[table]).as_posix()
            con.execute(f"""
                create or replace table {table} as
                select *, current_timestamp as _ingested_at, '{RAW_FILES[table]}' as _source_file
                from read_csv('{src}', all_varchar = true, header = true)
            """)
    missing_bronze = [t for t in BRONZE_TABLES if t not in existing_tables()]

assert not missing_bronze, f'Thiếu bảng Bronze: {missing_bronze}. Chạy 02_Bronze_Ingestion.ipynb trước.'
bronze_counts = pd.concat([query(f"select '{t}' as bang_bronze, count(*) as so_dong from {t}") for t in BRONZE_TABLES])
bronze_counts

## 3. C01 – Cấu hình dbt và `dbt debug`

`profiles.yml` thật không được commit. Notebook sao chép `profiles.yml.example` vào `~/.dbt/profiles.yml`;
đường dẫn DuckDB lấy từ biến môi trường `DUCKDB_PATH` đặt ở bước 1.

In [ ]:
profiles_dir = Path.home() / '.dbt'
profiles_dir.mkdir(exist_ok=True)
shutil.copy(DBT_DIR / 'profiles.yml.example', profiles_dir / 'profiles.yml')
print((profiles_dir / 'profiles.yml').read_text(encoding='utf-8'))

In [ ]:
_ = run_dbt('debug')

## 4. C02 – Xây dựng các model Silver

Mỗi model đọc một bảng Bronze, ép kiểu, chuẩn hóa chuỗi, xử lý giá trị thiếu/không hợp lệ và khử trùng theo grain.
`silver_products` phụ thuộc `silver_product_category_name_translation` (đã bổ sung danh mục thiếu bản dịch).

In [ ]:
_ = run_dbt('run', '--select', 'silver')

In [ ]:
SILVER_TABLES = ['silver_orders', 'silver_order_items', 'silver_order_payments', 'silver_order_reviews',
                 'silver_customers', 'silver_sellers', 'silver_products',
                 'silver_product_category_name_translation', 'silver_geolocation', 'silver_holidays']
schema_df = query("""
    select table_name as bang, count(*) as so_cot,
           string_agg(column_name || ' ' || data_type, ', ' order by ordinal_position) as cot_va_kieu
    from information_schema.columns
    where table_name like 'silver_%'
    group by table_name order by table_name
""")
schema_df

### 4.1. Đối soát số dòng Bronze → Silver

Silver chỉ được loại bản ghi trùng theo grain. Với `geolocation`, Silver gộp về 1 dòng / tiền tố ZIP.

In [ ]:
recon = query("""
    select 'orders' as bang, 'order_id' as grain,
           (select count(*) from bronze_orders) as so_dong_bronze,
           (select count(*) from silver_orders) as so_dong_silver
    union all select 'order_items', 'order_id + order_item_id',
           (select count(*) from bronze_order_items), (select count(*) from silver_order_items)
    union all select 'order_payments', 'order_id + payment_sequential',
           (select count(*) from bronze_order_payments), (select count(*) from silver_order_payments)
    union all select 'order_reviews', 'review_id + order_id',
           (select count(*) from bronze_order_reviews), (select count(*) from silver_order_reviews)
    union all select 'customers', 'customer_id',
           (select count(*) from bronze_customers), (select count(*) from silver_customers)
    union all select 'sellers', 'seller_id',
           (select count(*) from bronze_sellers), (select count(*) from silver_sellers)
    union all select 'products', 'product_id',
           (select count(*) from bronze_products), (select count(*) from silver_products)
    union all select 'product_category_name_translation', 'product_category_name',
           (select count(*) from bronze_product_category_name_translation),
           (select count(*) from silver_product_category_name_translation)
    union all select 'geolocation', 'geolocation_zip_code_prefix',
           (select count(*) from bronze_geolocation), (select count(*) from silver_geolocation)
    union all select 'holidays', 'holiday_date',
           (select count(*) from bronze_holidays), (select count(*) from silver_holidays)
""")
recon['chenh_lech'] = recon['so_dong_silver'] - recon['so_dong_bronze']
recon.to_csv(TABLE_DIR / 'Bang_3.4.3_DoiSoatSoDongBronzeSilver.csv', index=False, encoding='utf-8-sig')
recon

### 4.2. Đối soát giá trị tiền Bronze → Silver

In [ ]:
money = query("""
    select 'price' as chi_so,
           round((select sum(try_cast(price as double)) from bronze_order_items), 2) as tong_bronze,
           (select sum(price) from silver_order_items) as tong_silver
    union all select 'freight_value',
           round((select sum(try_cast(freight_value as double)) from bronze_order_items), 2),
           (select sum(freight_value) from silver_order_items)
    union all select 'payment_value',
           round((select sum(try_cast(payment_value as double)) from bronze_order_payments), 2),
           (select sum(payment_value) from silver_order_payments)
""")
money['tong_silver'] = money['tong_silver'].astype(float)
money['chenh_lech'] = (money['tong_silver'] - money['tong_bronze']).round(2)
money.to_csv(TABLE_DIR / 'Bang_3.4.4_DoiSoatGiaTriTien.csv', index=False, encoding='utf-8-sig')
money

### 4.3. Kết quả áp dụng quy tắc làm sạch

Số bản ghi chịu tác động của từng quy tắc, đếm trực tiếp trên bảng Silver (qua các cột cờ `flag_*`).

In [ ]:
impact = query("""
    select 'silver_orders' as bang, 'Đơn delivered nhưng thiếu ngày khách nhận' as quy_tac,
           count(*) filter (where flag_delivered_missing_date) as so_ban_ghi from silver_orders
    union all select 'silver_orders', 'Ngày giao vận chuyển trước thời điểm đặt (gắn cờ)',
           count(*) filter (where flag_carrier_before_purchase) from silver_orders
    union all select 'silver_orders', 'Ngày khách nhận trước ngày giao vận chuyển (gắn cờ)',
           count(*) filter (where flag_customer_before_carrier) from silver_orders
    union all select 'silver_orders', 'Đơn chưa có ngày khách nhận (giữ NULL, không xóa)',
           count(*) filter (where order_delivered_customer_date is null) from silver_orders
    union all select 'silver_order_payments', 'Số kỳ trả góp = 0 đưa về 1',
           count(*) filter (where flag_installments_imputed) from silver_order_payments
    union all select 'silver_order_payments', 'Dòng thanh toán giá trị 0 (giữ nguyên, gắn cờ)',
           count(*) filter (where flag_zero_payment) from silver_order_payments
    union all select 'silver_order_reviews', 'Đơn có nhiều hơn 1 đánh giá',
           (select count(*) from (select order_id from silver_order_reviews group by 1 having count(*) > 1))
    union all select 'silver_order_reviews', 'review_id dùng cho nhiều đơn',
           (select count(*) from (select review_id from silver_order_reviews group by 1 having count(*) > 1))
    union all select 'silver_order_reviews', 'Mốc thời gian sai định dạng được sửa',
           (select count(*) from bronze_order_reviews
             where not regexp_full_match(cast(review_answer_timestamp as varchar), '\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}')
                or not regexp_full_match(cast(review_creation_date as varchar), '\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}'))
    union all select 'silver_order_reviews', 'Bình luận chỉ có khoảng trắng chuyển thành NULL',
           (select count(*) from bronze_order_reviews
             where (review_comment_message is not null and trim(cast(review_comment_message as varchar)) = '')
                or (review_comment_title is not null and trim(cast(review_comment_title as varchar)) = ''))
    union all select 'silver_products', 'Danh mục NULL gán sem_categoria',
           count(*) filter (where flag_category_missing) from silver_products
    union all select 'silver_products', 'Sản phẩm thuộc danh mục được bổ sung bản dịch',
           count(*) filter (where product_category_name in ('pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos')) from silver_products
    union all select 'silver_products', 'Kích thước/khối lượng thiếu hoặc <= 0 (gán NULL)',
           count(*) filter (where flag_dimension_invalid) from silver_products
    union all select 'silver_sellers', 'Tên thành phố được chuẩn hóa',
           count(*) filter (where flag_city_standardized and not flag_city_invalid) from silver_sellers
    union all select 'silver_sellers', 'Tên thành phố không hợp lệ (số, email) chuyển NULL',
           count(*) filter (where flag_city_invalid) from silver_sellers
    union all select 'silver_customers', 'Tiền tố ZIP được khôi phục số 0 đầu',
           count(*) filter (where customer_zip_code_prefix like '0%') from silver_customers
    union all select 'silver_geolocation', 'Dòng Bronze được gộp (1.000.163 -> 1 dòng/ZIP)',
           (select count(*) from bronze_geolocation) - (select count(*) from silver_geolocation)
    union all select 'silver_geolocation', 'Điểm tọa độ ngoài lãnh thổ Brazil bị loại khỏi trung vị',
           sum(out_of_bounds_count) from silver_geolocation
    union all select 'silver_geolocation', 'ZIP không còn tọa độ hợp lệ (lat/lng NULL)',
           count(*) filter (where flag_no_valid_coordinate) from silver_geolocation
    union all select 'silver_geolocation', 'ZIP có nhiều bang ở nguồn (chọn bang phổ biến nhất)',
           count(*) filter (where flag_multiple_states) from silver_geolocation
""")
impact['so_ban_ghi'] = impact['so_ban_ghi'].astype(int)
impact.to_csv(TABLE_DIR / 'Bang_3.4.5_KetQuaQuyTacLamSach.csv', index=False, encoding='utf-8-sig')
impact

In [ ]:
# Chuẩn hóa tên thành phố: số giá trị phân biệt trước và sau
city = query("""
    select 'customers' as bang,
           (select count(distinct customer_city) from bronze_customers) as truoc,
           (select count(distinct customer_city) from silver_customers) as sau
    union all select 'sellers', (select count(distinct seller_city) from bronze_sellers),
                                (select count(distinct seller_city) from silver_sellers)
    union all select 'geolocation', (select count(distinct geolocation_city) from bronze_geolocation),
                                    (select count(distinct geolocation_city) from silver_geolocation)
""")
city

## 5. C03 – Kiểm thử tầng Silver

`dbt test --select silver` chạy toàn bộ test khai báo trong `models/silver/schema.yml` và các test tùy chỉnh
`tests/test_silver_*.sql`. Test ở mức **warn** ghi nhận bất thường có thật của dữ liệu nguồn (đã gắn cờ ở Silver),
không chặn pipeline.

In [ ]:
test_proc = run_dbt('test', '--select', 'silver', check=False)

In [ ]:
rr = json.loads((DBT_DIR / 'target' / 'run_results.json').read_text(encoding='utf-8'))

def test_group(name):
    if name.startswith('unique_combination'): return 'Khóa grain nhiều cột'
    if name.startswith('unique_'): return 'unique'
    if name.startswith('not_null_'): return 'not_null'
    if name.startswith('accepted_values_'): return 'accepted_values'
    if name.startswith('relationships_'): return 'relationships'
    return 'Tùy chỉnh (nghiệp vụ / đối soát)'

tests = pd.DataFrame([{
    'test': r['unique_id'].split('.')[2],
    'nhom': test_group(r['unique_id'].split('.')[2]),
    'trang_thai': r['status'].upper(),
    'so_dong_vi_pham': r.get('failures') or 0,
} for r in rr['results']]).sort_values(['nhom', 'test']).reset_index(drop=True)
tests['tang'] = 'silver'
tests['thoi_diem_chay'] = rr['metadata']['generated_at']
tests.to_csv(DQ_DIR / 'silver_test_results.csv', index=False, encoding='utf-8-sig')

summary = tests.pivot_table(index='nhom', columns='trang_thai', values='test', aggfunc='count', fill_value=0)
summary['tong'] = summary.sum(axis=1)
summary.loc['Tổng'] = summary.sum()
summary.to_csv(TABLE_DIR / 'Bang_3.4.6_TongHopKiemThuSilver.csv', encoding='utf-8-sig')
display(summary)
display(tests[tests['trang_thai'] != 'PASS'])
assert not (tests['trang_thai'].isin(['FAIL', 'ERROR'])).any(), 'Có test Silver FAIL/ERROR - kiểm tra log ở trên.'
print('Không có test FAIL/ERROR.')

## 6. Kết quả

- Bảng Silver nằm trong `commerceflow.duckdb` (schema `main`), sẵn sàng cho Task D02–D03 (Gold).
- Bảng số liệu cho báo cáo mục 3.4: `reports/tables/Bang_3.4.*.csv`.
- Kết quả kiểm thử Silver: `docs/data_quality/silver_test_results.csv` (đầu vào cho mục 3.7, 4.1 và `quality_results.parquet`).
- File `.duckdb` không commit lên GitHub; sao lưu theo quy tắc `commerceflow_YYYYMMDD.duckdb` nếu cần chuyển cho thành viên khác.

In [ ]:
print('Silver tables:')
display(recon[['bang', 'so_dong_silver']])
print('Output files:')
for p in sorted(TABLE_DIR.glob('Bang_3.4.*.csv')) + [DQ_DIR / 'silver_test_results.csv']:
    print(' -', p.relative_to(PROJECT_ROOT))